# Dataset 2 — Steel Industry Energy Consumption (UCI)
## Etapa B — Python / Pandas

Pré-requisito: Etapa A no Orange (Select Columns mantendo `Usage_kWh`, variáveis de potência reativa, fatores de potência, `WeekStatus`, `Day_of_week`, `Load_Type`; checar valores únicos e ausentes; amostra aleatória de 20%; exportar CSV). Ajuste `CAMINHO_CSV` para o arquivo exportado.

In [1]:
import pandas as pd

CAMINHO_CSV = "Steel_industry_data.csv"

df = pd.read_csv(CAMINHO_CSV)
df.columns

Index(['date', 'Usage_kWh', 'Lagging_Current_Reactive.Power_kVarh',
       'Leading_Current_Reactive_Power_kVarh', 'Lagging_Current_Power_Factor',
       'Leading_Current_Power_Factor', 'WeekStatus', 'Day_of_week',
       'Load_Type'],
      dtype='object')

### 1. Renomear Usage_kWh e simplificar fatores de potência

Os nomes originais de fator de potência no dataset UCI costumam ser `Lagging_Current_Power_Factor` e `Leading_Current_Power_Factor` — ajuste conforme os nomes reais das colunas mantidas na Etapa A.

In [2]:
df = df.rename(columns={
    "Usage_kWh": "Consumo_kWh",
    "Lagging_Current_Power_Factor": "Fator_Potencia_Atrasado",
    "Leading_Current_Power_Factor": "Fator_Potencia_Adiantado",
})
df.head()

,date,Consumo_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,Fator_Potencia_Atrasado,Fator_Potencia_Adiantado,WeekStatus,Day_of_week,Load_Type
0,29/04/2018 07:15,2.88,3.82,0.0,60.20,100.00,Weekend,Sunday,Light_Load
1,04/10/2018 12:00,60.77,48.02,0.0,78.46,100.00,Weekday,Thursday,Maximum_Load
2,26/01/2018 11:30,120.42,59.65,0.0,89.61,100.00,Weekday,Friday,Maximum_Load
3,02/06/2018 14:30,3.13,0.00,16.6,100.00,18.53,Weekend,Saturday,Light_Load
4,07/12/2018 15:00,58.86,20.99,0.0,94.19,100.00,Weekday,Friday,Medium_Load


### 2. Inspeção inicial: head(), shape, info(), describe()

In [3]:
df.head()

,date,Consumo_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,Fator_Potencia_Atrasado,Fator_Potencia_Adiantado,WeekStatus,Day_of_week,Load_Type
0,29/04/2018 07:15,2.88,3.82,0.0,60.20,100.00,Weekend,Sunday,Light_Load
1,04/10/2018 12:00,60.77,48.02,0.0,78.46,100.00,Weekday,Thursday,Maximum_Load
2,26/01/2018 11:30,120.42,59.65,0.0,89.61,100.00,Weekday,Friday,Maximum_Load
3,02/06/2018 14:30,3.13,0.00,16.6,100.00,18.53,Weekend,Saturday,Light_Load
4,07/12/2018 15:00,58.86,20.99,0.0,94.19,100.00,Weekday,Friday,Medium_Load


In [4]:
df.shape

(7008, 9)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7008 entries, 0 to 7007
Data columns (total 9 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   date                                  7008 non-null   object 
 1   Consumo_kWh                           7008 non-null   float64
 2   Lagging_Current_Reactive.Power_kVarh  7008 non-null   float64
 3   Leading_Current_Reactive_Power_kVarh  7008 non-null   float64
 4   Fator_Potencia_Atrasado               7008 non-null   float64
 5   Fator_Potencia_Adiantado              7008 non-null   float64
 6   WeekStatus                            7008 non-null   object 
 7   Day_of_week                           7008 non-null   object 
 8   Load_Type                             7008 non-null   object 
dtypes: float64(5), object(4)
memory usage: 492.9+ KB


In [6]:
df.describe()

,Consumo_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,Fator_Potencia_Atrasado,Fator_Potencia_Adiantado
count,7008.000000,7008.000000,7008.000000,7008.000000,7008.000000
mean,27.790164,13.206839,3.803711,80.672277,84.491448
std,33.717719,16.444246,7.356412,18.869855,30.314798
min,2.450000,0.000000,0.000000,37.870000,12.710000
25%,3.200000,2.380000,0.000000,63.602500,99.710000
50%,4.570000,5.040000,0.000000,87.990000,100.000000
75%,52.107500,22.977500,1.857500,98.842500,100.000000
max,153.140000,87.700000,27.650000,100.000000,100.000000


### 3. Maior consumo registrado e limiar de 75%

In [7]:
max_consumo = df["Consumo_kWh"].max()
limiar_75 = 0.75 * max_consumo

print(f"Consumo máximo: {max_consumo}")
print(f"Limiar (75% do máximo): {limiar_75}")

Consumo máximo: 153.14
Limiar (75% do máximo): 114.85499999999999


### 4. DataFrame de consumo acima do limiar: quantidade e percentual

In [8]:
df_consumo_alto = df[df["Consumo_kWh"] > limiar_75]

qtd_consumo_alto = len(df_consumo_alto)
percentual_consumo_alto = qtd_consumo_alto / len(df) * 100

print(f"Registros de consumo elevado: {qtd_consumo_alto}")
print(f"Percentual sobre o total da amostra: {percentual_consumo_alto:.2f}%")

Registros de consumo elevado: 129
Percentual sobre o total da amostra: 1.84%


### 5. Quantos desses registros pertencem à categoria Maximum Load

In [9]:
qtd_maximum_load = (df_consumo_alto["Load_Type"] == "Maximum_Load").sum()
percentual_maximum_load = qtd_maximum_load / qtd_consumo_alto * 100

print(f"Registros com Load_Type == 'Maximum_Load' dentro do consumo elevado: {qtd_maximum_load}")
print(f"Percentual dentro do consumo elevado: {percentual_maximum_load:.2f}%")

Registros com Load_Type == 'Maximum_Load' dentro do consumo elevado: 62
Percentual dentro do consumo elevado: 48.06%


### 6. Limite coerente para fator de potência baixo

Observe a distribuição do fator de potência escolhido antes de fixar o limite — o valor abaixo (`0.80`) é um ponto de partida comum para "fator de potência baixo" em indústrias, mas deve ser ajustado após olhar `describe()`/histograma da coluna real.

In [10]:
df["Fator_Potencia_Atrasado"].describe()

,Fator_Potencia_Atrasado
count,7008.000000
mean,80.672277
std,18.869855
min,37.870000
25%,63.602500
50%,87.990000
75%,98.842500
max,100.000000


In [11]:
limite_fator_potencia_baixo = 0.80
limite_fator_potencia_baixo

0.8

### 7. DataFrame com consumo elevado E fator de potência abaixo do limite

In [12]:
df_consumo_fp_baixo = df[
    (df["Consumo_kWh"] > limiar_75) &
    (df["Fator_Potencia_Atrasado"] < limite_fator_potencia_baixo)
]

qtd_consumo_fp_baixo = len(df_consumo_fp_baixo)
percentual_consumo_fp_baixo = qtd_consumo_fp_baixo / len(df) * 100

print(f"Registros com consumo elevado E fator de potência baixo: {qtd_consumo_fp_baixo}")
print(f"Percentual sobre o total da amostra: {percentual_consumo_fp_baixo:.2f}%")

Registros com consumo elevado E fator de potência baixo: 0
Percentual sobre o total da amostra: 0.00%


**Interpretação (preencher com os valores impressos acima antes de entregar):**

Esse segundo conjunto (consumo elevado + fator de potência baixo) merece mais atenção da equipe de energia porque combina dois problemas ao mesmo tempo: alta demanda de potência ativa **e** ineficiência no uso dessa energia. Um fator de potência baixo indica que parte da energia consumida da rede não está sendo convertida em trabalho útil (é potência reativa "desperdiçada" no sistema), o que geralmente implica:

- Maior custo, já que concessionárias costumam penalizar fator de potência abaixo de um limite contratual.
- Maior estresse na rede elétrica da planta durante justamente os picos de consumo, aumentando o risco de sobrecarga.
- Uma oportunidade concreta de intervenção (bancos de capacitores, correção de fator de potência) que tem retorno financeiro direto, diferente de apenas reduzir consumo bruto.

Consumo alto isolado pode ser simplesmente uma operação normal em carga máxima; consumo alto combinado com fator de potência baixo é o sinal de que há ineficiência a corrigir, não apenas demanda a gerenciar.